# 07-5. HTTP 오류·타임아웃·재시도 예제

## Goal

- 상태 코드별 재시도 여부를 구분합니다.
- 서버의 Retry-After 값을 제한해서 사용합니다.

이 노트북은 교안 예제를 안전하게 재현하는 보조 실습입니다. 먼저 결과를 예측한 뒤 셀을 실행하세요.


## Setup

실제 요청과 대기는 수행하지 않고 정책만 계산합니다.


## Steps

### HTTP 재시도 계획 만들기

읽기 전용 요청에서도 무제한 재시도하지 않으며 클라이언트 오류는 기본적으로 즉시 중단합니다.


In [1]:
RETRYABLE_STATUS = {429, 502, 503, 504}


def retry_decision(method: str, status: int, retry_after: str | None, attempt: int, limit=3):
    if method.upper() not in {"GET", "HEAD"}:
        return {"retry": False, "reason": "method"}
    if status not in RETRYABLE_STATUS or attempt >= limit:
        return {"retry": False, "reason": "status_or_limit"}
    delay = min(float(retry_after), 10.0) if retry_after and retry_after.isdigit() else 0.5 * (2 ** (attempt - 1))
    return {"retry": True, "delay": delay}


scenarios = [
    ("GET", 503, None, 1),
    ("GET", 429, "20", 2),
    ("POST", 503, None, 1),
    ("GET", 404, None, 1),
]
decisions = [retry_decision(*scenario) for scenario in scenarios]
for scenario, decision in zip(scenarios, decisions):
    print(scenario, "->", decision)


('GET', 503, None, 1) -> {'retry': True, 'delay': 0.5}
('GET', 429, '20', 2) -> {'retry': True, 'delay': 10.0}
('POST', 503, None, 1) -> {'retry': False, 'reason': 'method'}
('GET', 404, None, 1) -> {'retry': False, 'reason': 'status_or_limit'}


## Checks

서버 지시 대기 시간의 상한과 비멱등 메서드 거부를 확인합니다.


In [2]:
assert decisions[0] == {"retry": True, "delay": 0.5}
assert decisions[1] == {"retry": True, "delay": 10.0}
assert decisions[2]["retry"] is False
assert decisions[3]["retry"] is False
print("재시도 정책 검사 통과")


재시도 정책 검사 통과


## Next Steps

실제 코드에서는 연결·읽기 타임아웃과 전체 실행 시간 한도를 함께 기록합니다.
